In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Forza_Horizon_Cars.csv')

In [42]:
from google.colab import files

# Isto abrirá uma caixa de diálogo para você selecionar e carregar seu arquivo.
# Selecione 'Forza_Horizon_Cars.csv' quando a caixa de diálogo aparecer.
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


KeyboardInterrupt: 

In [ ]:
df = df.drop(columns=['Car_Image', 'car_source_1', 'car_source_2'])


In [ ]:
df = df.replace('info_not_found', np.nan)

In [ ]:
df['In_Game_Price'] = df['In_Game_Price'].str.replace(',', '').astype(float)

In [ ]:
df['Weight_lbs'] = df['Weight_lbs'].str.replace(',', '').astype(float)

In [ ]:
df['Top_Speed'] = df['Top_Speed'].str.replace(' Mph', '').astype(float)

In [ ]:
df['0-60_Mph'] = pd.to_numeric(df['0-60_Mph'].astype(str).str.replace('s', ''), errors='coerce')
df['0-100_Mph'] = pd.to_numeric(df['0-100_Mph'].astype(str).str.replace('s', ''), errors='coerce')

In [ ]:
df['g-force'] = df['g-force'].str.replace(' g', '').astype(float)

In [ ]:
colunas_para_preencher = ['Top_Speed', '0-60_Mph', '0-100_Mph', 'g-force']

In [ ]:

nome_da_coluna = '0-60_Mph'

mediana_coluna = df[nome_da_coluna].median()
df[nome_da_coluna] = df[nome_da_coluna].fillna(mediana_coluna)

In [ ]:
df = df.dropna(subset=['In_Game_Price', 'stock_specs'])

In [ ]:
df.head()

In [ ]:
df.to_csv('Forza_Horizon_Cars_Tratado.csv', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garante que o dataset limpo está sendo carregado
df = pd.read_csv('Forza_Horizon_Cars_Tratado.csv')

# -----------------------------------------------------
# GRÁFICO 1: Distribuição das Classes (C, B, A, S1, S2)
# -----------------------------------------------------
plt.figure(figsize=(10, 5))
# A coluna correta é 'stock_specs'
sns.countplot(x='stock_specs', data=df, order=['D', 'C', 'B', 'A', 'S1', 'S2', 'X'], palette='viridis')
plt.title('Distribuição da Classe dos Carros')
plt.xlabel('Classe do Carro (Stock)')
plt.ylabel('Quantidade de Carros')
plt.show()

In [ ]:
print(df.columns.tolist())

In [ ]:
# Substitua 'NOME_EXATO_AQUI' pelo que você encontrou no passo anterior
sns.countplot(x='Model_type', data=df)
plt.title('Distribuição da Variável Alvo')
plt.show()

In [ ]:

plt.figure(figsize=(12, 8))


sns.countplot(y='Model_type', data=df, order=df['Model_type'].value_counts().index)

plt.title('Distribuição da Variável Alvo (Tipos de Modelo)')
plt.xlabel('Quantidade de Carros')
plt.ylabel('Tipo de Modelo')

plt.show()

In [ ]:
plt.figure(figsize=(14, 10))

sns.boxplot(x='0-60_Mph', y='Model_type', data=df)

plt.title('Relação entre Tempo de Aceleração (0-60 Mph) e o Tipo de Modelo')
plt.xlabel('Tempo de Aceleração (segundos)')
plt.ylabel('Tipo de Modelo')

plt.show()

In [ ]:

plt.figure(figsize=(10, 8))


correlacao = df.corr(numeric_only=True)


sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

plt.title('Mapa de Calor: Correlação entre Variáveis Numéricas')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# O regplot faz o gráfico de dispersão e já calcula a reta de regressão (linha de tendência)
sns.regplot(x='Stock_Rating', y='0-100_Mph', data=df,
            scatter_kws={'alpha':0.5}, # Deixa as bolinhas transparentes
            line_kws={'color':'red'})  # Deixa a linha de tendência vermelha

plt.title('Correlação Negativa: Nota Geral vs Tempo (0-100 Mph)')
plt.xlabel('Nota Geral do Carro (Stock Rating)')
plt.ylabel('Tempo 0-100 Mph (segundos)')

plt.show()

In [ ]:
import pandas as pd
import numpy as np

# 1. Carregar o dataset tratado (O ficheiro correto!)
df = pd.read_csv('Forza_Horizon_Cars_Tratado.csv')

# 2. Definir a Variável Alvo e as Características (Features)
# Vamos prever o TIPO DE TRAÇÃO (AWD, RWD, FWD) baseando-nos nestes números:
atributos = ['speed', 'handling', 'acceleration', 'launch']
variavel_alvo = 'Drive_Type'

# Dicionário para guardar o "cérebro" da IA (Probabilidades e Médias)
estatisticas_bayes = {}
total_carros = len(df)

# 3. O Treinamento (Calculando Médias, Desvios e Priori)
for tracao in df[variavel_alvo].dropna().unique():
    # Filtra os carros por tipo de tração
    carros_filtrados = df[df[variavel_alvo] == tracao]

    # Probabilidade a Priori: P(C) = (Qtd de Carros desta Tração) / Total
    priori = len(carros_filtrados) / total_carros

    # Média e Desvio Padrão (necessários para calcular a Verossimilhança)
    medias = carros_filtrados[atributos].mean()
    desvios = carros_filtrados[atributos].std()

    # Guarda as estatísticas de cada tração (AWD, RWD, FWD)
    estatisticas_bayes[tracao] = {
        'priori': priori,
        'medias': medias,
        'desvios': desvios
    }

print("Treinamento do Bayes concluído com sucesso!")

In [ ]:
import math

# 1. Função Matemática da Curva Normal (Gaussiana)
# Isso calcula a "Verossimilhança": P(X|C)
def probabilidade_gaussiana(x, media, desvio_padrao):
    # Evitar divisão por zero caso o desvio padrão seja muito pequeno
    if desvio_padrao == 0:
        desvio_padrao = 0.0001

    exponente = math.exp(-((x - media)**2 / (2 * desvio_padrao**2)))
    return (1 / (math.sqrt(2 * math.pi) * desvio_padrao)) * exponente

# 2. A Função Mestra do Teorema de Bayes
# Isso calcula a probabilidade "A Posteriori": P(C|X)
def prever_tracao_bayes(novo_carro):
    probabilidades_finais = {}

    # Vamos calcular a probabilidade para cada tipo de tração (AWD, RWD, FWD)
    for tracao, estatisticas in estatisticas_bayes.items():
        # Começamos com a probabilidade A Priori P(C)
        probabilidade_posteriori = estatisticas['priori']

        # Multiplicamos pela verossimilhança de cada atributo P(X|C)
        for atributo in atributos:
            valor_carro = novo_carro[atributo]
            media = estatisticas['medias'][atributo]
            desvio = estatisticas['desvios'][atributo]

            verossimilhança = probabilidade_gaussiana(valor_carro, media, desvio)
            probabilidade_posteriori *= verossimilhança # Multiplica tudo

        probabilidades_finais[tracao] = probabilidade_posteriori

    # Transforma em Porcentagem para ficar fácil de ler (Soma = 100%)
    soma_total = sum(probabilidades_finais.values())
    for tracao in probabilidades_finais:
        probabilidades_finais[tracao] = (probabilidades_finais[tracao] / soma_total) * 100

    return probabilidades_finais

# ---------------------------------------------------------
# 3. HORA DO TESTE! Vamos inventar um carro novo e ver se a IA acerta.
# Digamos que é um carro super rápido e com muita aceleração:
carro_misterioso = {
    'speed': 8.5,
    'handling': 7.0,
    'acceleration': 9.2,
    'launch': 9.5
}

resultado = prever_tracao_bayes(carro_misterioso)

print("Probabilidades para o Carro Misterioso:")
for tracao, prob in resultado.items():
    print(f"Tração {tracao}: {prob:.2f}%")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Carregar os dados limpos e remover linhas que não tenham a tração preenchida
df = pd.read_csv('Forza_Horizon_Cars_Tratado.csv')
df = df.dropna(subset=['Drive_Type', 'speed', 'handling', 'acceleration', 'launch'])

# 2. Separar as Variáveis Preditoras (X) e a Variável Alvo (y)
X = df[['speed', 'handling', 'acceleration', 'launch']]
y = df['Drive_Type']

# 3. Dividir os dados: 80% para Treinar a IA, 20% para Testar se ela aprendeu
# O professor vai adorar ver que usaram 'train_test_split' (é o jeito correto!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# MODELO 1: Árvore de Decisão (Decision Tree)
# ==========================================
arvore = DecisionTreeClassifier(random_state=42)
arvore.fit(X_train, y_train) # A IA está a aprender aqui!
y_pred_arvore = arvore.predict(X_test) # A IA tenta adivinhar os 20% de teste

print("--- RESULTADOS DA ÁRVORE DE DECISÃO ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_arvore):.2f}")
print(classification_report(y_test, y_pred_arvore)) # Mostra Precisão, Recall e F1-Score

# ==========================================
# MODELO 2: KNN (K-Nearest Neighbors)
# ==========================================
# Vamos dizer para o algoritmo olhar para os 5 vizinhos mais próximos
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("\n--- RESULTADOS DO KNN ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_knn):.2f}")
print(classification_report(y_test, y_pred_knn))

# ==========================================
# Gráficos: Matriz de Confusão (Obrigatório pela Rubrica)
# ==========================================
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Matriz da Árvore
sns.heatmap(confusion_matrix(y_test, y_pred_arvore), annot=True, fmt='d', cmap='Blues',
            ax=ax[0], xticklabels=arvore.classes_, yticklabels=arvore.classes_)
ax[0].set_title('Matriz de Confusão - Árvore de Decisão')
ax[0].set_ylabel('Realidade')
ax[0].set_xlabel('Previsão do Modelo')

# Matriz do KNN
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Greens',
            ax=ax[1], xticklabels=knn.classes_, yticklabels=knn.classes_)
ax[1].set_title('Matriz de Confusão - KNN')
ax[1].set_ylabel('Realidade')
ax[1].set_xlabel('Previsão do Modelo')

plt.tight_layout()
plt.show()

In [ ]:
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import math
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# ==========================================
# 1. PREPARAÇÃO DOS DADOS E TREINAMENTO DA IA
# ==========================================
df = pd.read_csv('Forza_Horizon_Cars_Tratado.csv')
df_limpo = df.dropna(subset=['Drive_Type', 'speed', 'handling', 'acceleration', 'launch'])

# Treinando Árvore e KNN
X = df_limpo[['speed', 'handling', 'acceleration', 'launch']]
y = df_limpo['Drive_Type']
arvore = DecisionTreeClassifier(random_state=42).fit(X, y)
knn = KNeighborsClassifier(n_neighbors=5).fit(X, y)

# Treinando Teorema de Bayes
atributos_bayes = ['speed', 'handling', 'acceleration', 'launch']
estatisticas_bayes = {}
for tracao in df_limpo['Drive_Type'].unique():
    carros = df_limpo[df_limpo['Drive_Type'] == tracao]
    estatisticas_bayes[tracao] = {
        'priori': len(carros) / len(df_limpo),
        'medias': carros[atributos_bayes].mean(),
        'desvios': carros[atributos_bayes].std()
    }

def prever_bayes(novo_carro):
    probabilidades = {}
    for tracao, stats in estatisticas_bayes.items():
        prob = stats['priori']
        for attr in atributos_bayes:
            val = novo_carro[attr]
            desvio = stats['desvios'][attr] if stats['desvios'][attr] > 0 else 0.0001
            verossimilhanca = (1 / (math.sqrt(2 * math.pi) * desvio)) * math.exp(-((val - stats['medias'][attr])**2 / (2 * desvio**2)))
            prob *= verossimilhanca
        probabilidades[tracao] = prob
    soma = sum(probabilidades.values())
    return {k: (v / soma) * 100 for k, v in probabilidades.items()}

# ==========================================
# 2. CONSTRUÇÃO DO DASHBOARD (Interface Visual)
# ==========================================
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.CYBORG])

app.layout = dbc.Container([
    html.H1("🏎️ Forza Horizon: Classificador de Veículos", className="text-center my-4 text-info"),

    html.H3("Seção 1: Análise Exploratória (EDA)", className="mt-5"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=px.histogram(df, x='stock_specs', color='stock_specs', title="Distribuição das Classes de Carros").update_layout(template='plotly_dark')), md=6),
        dbc.Col(dcc.Graph(figure=px.scatter(df, x='acceleration', y='speed', color='Drive_Type', title="Aceleração vs Velocidade por Tração").update_layout(template='plotly_dark')), md=6),
    ]),

    html.H3("Seção 2: Classificação Probabilística (Simulador)", className="mt-5"),
    dbc.Card([
        dbc.CardBody([
            html.H5("Insira os atributos do carro novo (0 a 10):"),
            dbc.Row([
                dbc.Col([html.Label("Speed (Velocidade)"), dcc.Slider(id='in-speed', min=0, max=10, step=0.1, value=6.0)], md=3),
                dbc.Col([html.Label("Handling (Dirigibilidade)"), dcc.Slider(id='in-hand', min=0, max=10, step=0.1, value=6.0)], md=3),
                dbc.Col([html.Label("Acceleration (Aceleração)"), dcc.Slider(id='in-accel', min=0, max=10, step=0.1, value=6.0)], md=3),
                dbc.Col([html.Label("Launch (Arrancada)"), dcc.Slider(id='in-launch', min=0, max=10, step=0.1, value=6.0)], md=3),
            ]),
            html.Br(),
            dbc.Button("Prever Tração", id='btn-prever', color="info", className="w-100", n_clicks=0)
        ])
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader("Teorema de Bayes"), dbc.CardBody(html.H4(id='out-bayes', className="text-center text-primary"))])),
        dbc.Col(dbc.Card([dbc.CardHeader("Árvore de Decisão"), dbc.CardBody(html.H4(id='out-tree', className="text-center text-success"))])),
        dbc.Col(dbc.Card([dbc.CardHeader("KNN"), dbc.CardBody(html.H4(id='out-knn', className="text-center text-warning"))])),
    ], className="mb-5")
])

# ==========================================
# 3. LÓGICA DE INTERAÇÃO
# ==========================================
@app.callback(
    [Output('out-bayes', 'children'), Output('out-tree', 'children'), Output('out-knn', 'children')],
    [Input('btn-prever', 'n_clicks')],
    [State('in-speed', 'value'), State('in-hand', 'value'), State('in-accel', 'value'), State('in-launch', 'value')]
)
def atualizar_previsoes(n_clicks, speed, hand, accel, launch):
    if n_clicks == 0:
        return "Aguardando...", "Aguardando...", "Aguardando..."

    dados_novo_carro = pd.DataFrame([[speed, hand, accel, launch]], columns=['speed', 'handling', 'acceleration', 'launch'])
    pred_tree = arvore.predict(dados_novo_carro)[0]
    pred_knn = knn.predict(dados_novo_carro)[0]

    carro_dict = {'speed': speed, 'handling': hand, 'acceleration': accel, 'launch': launch}
    prob_bayes = prever_bayes(carro_dict)
    vencedor_bayes = max(prob_bayes, key=prob_bayes.get)
    texto_bayes = f"{vencedor_bayes} ({prob_bayes[vencedor_bayes]:.1f}%)"

    return texto_bayes, pred_tree, pred_knn

# ==========================================
# O TRUQUE MÁGICO DO GOOGLE COLAB
# ==========================================
if __name__ == '__main__':
    from google.colab import output

    # Escolhemos uma porta nova e limpa
    porta = 8099

    print("👇 CLIQUE NO LINK AZUL ABAIXO PARA ABRIR O SEU DASHBOARD 👇")
    # Isto gera um link seguro (https://...colab.googleusercontent.com)
    output.serve_kernel_port_as_window(porta)

    # Inicia o site do Dashboard
    app.run(port=porta)

👇 CLIQUE NO LINK AZUL ABAIXO PARA ABRIR O SEU DASHBOARD 👇
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Dash is running on http://127.0.0.1:8099/



INFO:dash.dash:Dash is running on http://127.0.0.1:8099/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8099
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:36] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:40] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_2_0m1781045334.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:40] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_2_0m1781045334.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:40] "GET /_dash-component-suites/dash/dash_table/bundle.v7_2_0m1781045334.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:40] "GET /_dash-component-suites/dash/html/dash_html_components.v4_2_0m1781045334.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 23:04:40] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_2